# Shap-E

Objective: to generate 3D objects conditioned on text or images.

Reference: https://github.com/openai/shap-e

## Install Python packages

In [ ]:
!pip install shap-e torch torchvision numpy fastapi uvicorn pyngrok nest_asyncio pydantic

## Prepare the models

In [ ]:
import torch

from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import create_pan_cameras, decode_latent_images, gif_widget

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

xm = load_model('transmitter', device=device)
model = load_model('text300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))
# Shap-E imports (keep yours)
import torch
import numpy as np

# ADD THESE
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import FileResponse
import tempfile
import os
import threading
import nest_asyncio
from pyngrok import ngrok
import uvicorn

## Define a wrapper

In [ ]:
def sample_text_to_3d(prompt):
    batch_size = 1
    guidance_scale = 15.0

    latents = diffusion.sample(
        model,
        batch_size=batch_size,
        model_kwargs=dict(texts=[prompt]),
        guidance_scale=guidance_scale,
        device=device,
    )

    return latents[0]

## Sample a 3D model

### Conditioned on a text prompt

In [ ]:

def save_model(latent, file_path):
    from shap_e.util.notebooks import decode_latent_mesh

    mesh = decode_latent_mesh(xm, latent).tri_mesh()

    with open(file_path, "wb") as f:
        mesh.write_glb(f)

Fast API


In [ ]:
app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str

@app.post("/generate_shap_e")
def generate_model(req: PromptRequest):
    prompt = req.prompt

    temp_dir = tempfile.mkdtemp()
    file_path = os.path.join(temp_dir, "model.glb")

    model = sample_text_to_3d(prompt)
    save_model(model, file_path)

    return FileResponse(
        file_path,
        media_type="model/gltf-binary",
        filename="model.glb"
    )

NGROK

In [ ]:
nest_asyncio.apply()

ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # <-- PUT YOUR TOKEN HERE

public_url = ngrok.connect(8000)

print("PUBLIC URL:", public_url)

In [ ]:
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

threading.Thread(target=server.run, daemon=True).start()

In [ ]:
import requests

r = requests.post(
    str(public_url) + "/generate_shap_e",
    json={"prompt": "low poly wooden chair"}
)

print(r.status_code)
print(r.headers.get("content-type"))